In [17]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("dataset_research")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "Run this notebook from the Assignment3 folder containing dataset_research."
    )

In [18]:
bike = pd.read_csv(DATA_DIR / "bike-sharing" / "day.csv", parse_dates=["dteday"])
bike = bike.set_index("dteday").sort_index()
bike.index.name = "date"

print(f"Capital Bikeshare: {bike.shape[0]:,} rows, {bike.shape[1]} columns")
display(bike.head())

Capital Bikeshare: 731 rows, 15 columns


,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
date,,,,,,,,,,,,,,,
2011-01-01,1,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
2011-01-02,2,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2011-01-03,3,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
2011-01-04,4,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
2011-01-05,5,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [19]:
sunspots = pd.read_csv(DATA_DIR / "sunspots.csv")
sunspots.index = pd.date_range("1749-01-01", periods=len(sunspots), freq="MS", name="date")
sunspots = sunspots.drop(columns=["rownames", "time"])

print(f"Sunspots: {sunspots.shape[0]:,} rows, {sunspots.shape[1]} column")
display(sunspots.head())

Sunspots: 2,820 rows, 1 column


,value
date,
1749-01-01,58.0
1749-02-01,62.6
1749-03-01,70.0
1749-04-01,55.7
1749-05-01,85.0


In [20]:
electricity = pd.read_csv(DATA_DIR / "usmelec.csv")
electricity.index = pd.date_range(
    "1973-01-01", periods=len(electricity), freq="MS", name="date"
)
electricity = electricity.drop(columns=["rownames", "time"])

print(f"US electricity: {electricity.shape[0]:,} rows, {electricity.shape[1]} column")
display(electricity.head())

US electricity: 486 rows, 1 column


,value
date,
1973-01-01,160.218
1973-02-01,143.539
1973-03-01,148.158
1973-04-01,139.589
1973-05-01,147.395


In [21]:
gasoline = pd.read_csv(DATA_DIR / "gasoline.csv")
gasoline = gasoline.drop(columns="rownames").set_index("time")
gasoline.index.name = "fractional_year"

print(f"US gasoline: {gasoline.shape[0]:,} rows, {gasoline.shape[1]} column")
display(gasoline.head())

US gasoline: 1,355 rows, 1 column


,value
fractional_year,
1991.100000,6.621
1991.119165,6.433
1991.138330,6.582
1991.157495,7.224
1991.176660,6.875


In [22]:
ett = pd.read_csv(DATA_DIR / "ETTh1.csv", parse_dates=["date"])
ett = ett.set_index("date").sort_index()

print(f"ETTh1: {ett.shape[0]:,} rows, {ett.shape[1]} columns")
display(ett.head())

ETTh1: 17,420 rows, 7 columns


,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


In [23]:
datasets = {
    "Bikeshare": bike,
    "Sunspots": sunspots,
    "US electricity": electricity,
    "US gasoline": gasoline,
    "ETTh1": ett,
}
target_columns = {
    "Bikeshare": "cnt",
    "Sunspots": "value",
    "US electricity": "value",
    "US gasoline": "value",
    "ETTh1": "OT",
}
frequencies = {
    "Bikeshare": "Daily",
    "Sunspots": "Monthly",
    "US electricity": "Monthly",
    "US gasoline": "Weekly",
    "ETTh1": "Hourly",
}
targets = {
    name: frame[target_columns[name]].rename(name)
    for name, frame in datasets.items()
}

In [24]:
import copy
import json
import random
import time
import warnings

import numpy as np
import torch
from statsmodels.tsa.stattools import adfuller, kpss
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


CONFIG = {
    "hidden_sizes": [8, 16],
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "batch_size": 64,
    "max_epochs": 60,
    "patience": 8,
    "clip_norm": 1.0,
    "development_fraction": 0.8,
    "fold_fractions": [0.55, 0.70, 0.85, 1.0],
    "tuning_seed": 42,
    "final_seeds": [11, 22, 33],
    "num_threads": 2,
}
LOOKBACKS = {
    "Bikeshare": 14,
    "Sunspots": 24,
    "US electricity": 24,
    "US gasoline": 13,
    "ETTh1": 24,
}
MODEL_NAMES = ("elman", "jordan", "multi")
DEVICE = torch.device("cpu")
torch.set_num_threads(CONFIG["num_threads"])
set_seed(CONFIG["tuning_seed"])
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
series_index = {}
series_values = {}
dev_ends = {}
audit_rows = []
expected_counts = {
    "Bikeshare": 731, "Sunspots": 2820, "US electricity": 486,
    "US gasoline": 1355, "ETTh1": 17420,
}

for name, series in targets.items():
    values = series.to_numpy(dtype=float, copy=True)
    index = series.index.copy()
    n = len(values)
    if n != expected_counts[name] or not np.isfinite(values).all():
        raise ValueError(f"{name}: incorrect row count or non-finite target values")
    if index.isna().any() or index.has_duplicates or not index.is_monotonic_increasing:
        raise ValueError(f"{name}: invalid, duplicate, or unordered time index")
    if name == "Bikeshare":
        regular = bool(((index[1:] - index[:-1]) == pd.Timedelta(days=1)).all())
    elif name in ("Sunspots", "US electricity"):
        regular = index.equals(pd.date_range(index[0], periods=n, freq="MS"))
    elif name == "US gasoline":
        regular = bool(np.allclose(np.diff(index.to_numpy(dtype=float)),
                                   7 / 365.25, rtol=0, atol=1e-7))
    else:
        regular = bool(((index[1:] - index[:-1]) == pd.Timedelta(hours=1)).all())
    if not regular:
        raise ValueError(f"{name}: irregular time index")
    D = int(CONFIG["development_fraction"] * n)
    if not LOOKBACKS[name] < D < n:
        raise ValueError(f"{name}: insufficient development or test history")
    series_index[name] = index
    series_values[name] = values
    dev_ends[name] = D
    audit_rows.append({
        "dataset": name, "observations": n, "frequency": frequencies[name],
        "first": str(index[0]), "last": str(index[-1]),
        "missing_values": int(np.isnan(values).sum()),
        "duplicate_times": int(index.duplicated().sum()),
        "regular_spacing": regular, "development": D, "test": n - D,
        "last_development": str(index[D - 1]), "first_test": str(index[D]),
    })

data_audit = pd.DataFrame(audit_rows)
display(data_audit)
data_audit.to_csv(OUTPUT_DIR / "data_audit.csv", index=False)

,dataset,observations,frequency,first,last,missing_values,duplicate_times,regular_spacing,development,test,last_development,first_test
0,Bikeshare,731,Daily,2011-01-01 00:00:00,2012-12-31 00:00:00,0,0,True,584,147,2012-08-06 00:00:00,2012-08-07 00:00:00
1,Sunspots,2820,Monthly,1749-01-01 00:00:00,1983-12-01 00:00:00,0,0,True,2256,564,1936-12-01 00:00:00,1937-01-01 00:00:00
2,US electricity,486,Monthly,1973-01-01 00:00:00,2013-06-01 00:00:00,0,0,True,388,98,2005-04-01 00:00:00,2005-05-01 00:00:00
3,US gasoline,1355,Weekly,1991.1,2017.04934976044,0,0,True,1084,271,2011.85564681725,2011.87481177276
4,ETTh1,17420,Hourly,2016-07-01 00:00:00,2018-06-26 19:00:00,0,0,True,13936,3484,2018-02-01 15:00:00,2018-02-01 16:00:00


In [26]:
def stationarity_summary(values):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or len(values) < 30 or not np.isfinite(values).all():
        raise ValueError("Stationarity tests require at least 30 finite values")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        adf = adfuller(values, regression="c", autolag="AIC")
        kpss_result = kpss(values, regression="c", nlags="auto")
    return {
        "adf_stat": float(adf[0]), "adf_p": float(adf[1]),
        "adf_lags": int(adf[2]), "adf_nobs": int(adf[3]),
        "kpss_stat": float(kpss_result[0]), "kpss_p": float(kpss_result[1]),
        "kpss_lags": int(kpss_result[2]),
        "warnings": " | ".join(str(item.message) for item in caught),
    }


rolling_windows = {
    "Bikeshare": 30, "Sunspots": 132, "US electricity": 12,
    "US gasoline": 52, "ETTh1": 168,
}
stationarity_rows = []
for name, values in series_values.items():
    D = dev_ends[name]
    development = values[:D]
    index = series_index[name][:D]
    result = stationarity_summary(development)
    adf_rejects = result["adf_p"] < 0.05
    kpss_rejects = result["kpss_p"] < 0.05
    if adf_rejects and not kpss_rejects:
        judgment = "stationary by ADF/KPSS at 5%"
    elif not adf_rejects and kpss_rejects:
        judgment = "non-stationary by ADF/KPSS at 5%"
    elif adf_rejects and kpss_rejects:
        judgment = "non-stationary: changing level; ADF and KPSS disagree"
    else:
        judgment = "tests inconclusive; inspect trend/seasonality"
    stationarity_rows.append({
        "dataset": name, "development_observations": D,
        **result, "test_based_judgment": judgment,
    })
    rolling = pd.Series(development, index=index).rolling(rolling_windows[name])
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(index, development, label="target", linewidth=0.8)
    ax.plot(index, rolling.mean(), label="rolling mean", linewidth=1)
    ax.plot(index, rolling.std(), label="rolling SD", linewidth=1)
    ax.set_title(f"{name}: development period only")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"stationarity_{name.lower().replace(' ', '_')}.png", dpi=130)
    plt.close(fig)

stationarity_results = pd.DataFrame(stationarity_rows)
display(stationarity_results)
stationarity_results.to_csv(OUTPUT_DIR / "stationarity_results.csv", index=False)

,dataset,development_observations,adf_stat,adf_p,adf_lags,adf_nobs,kpss_stat,kpss_p,kpss_lags,warnings,test_based_judgment
0,Bikeshare,584,-1.542329,5.124775e-01,10,573,2.100955,0.01,15,adfuller currently returns a plain tuple whose...,non-stationary by ADF/KPSS at 5%
1,Sunspots,2256,-7.939288,3.347484e-12,26,2229,0.176058,0.10,28,adfuller currently returns a plain tuple whose...,stationary by ADF/KPSS at 5%
2,US electricity,388,-0.134143,9.459298e-01,15,372,3.301170,0.01,11,adfuller currently returns a plain tuple whose...,non-stationary by ADF/KPSS at 5%
3,US gasoline,1084,-2.370380,1.502714e-01,8,1075,4.988013,0.01,19,adfuller currently returns a plain tuple whose...,non-stationary by ADF/KPSS at 5%
4,ETTh1,13936,-3.180623,2.113406e-02,42,13893,7.462117,0.01,71,adfuller currently returns a plain tuple whose...,non-stationary: changing level; ADF and KPSS d...


In [27]:
def make_folds(dev_end):
    endpoints = [int(dev_end * fraction) for fraction in CONFIG["fold_fractions"]]
    if not 0 < endpoints[0] < endpoints[1] < endpoints[2] < endpoints[3] == dev_end:
        raise ValueError("Development series is too short for the chosen folds")
    return [
        {"fold": number, "train_end": endpoints[number - 1],
         "val_end": endpoints[number]}
        for number in range(1, len(endpoints))
    ]

In [28]:
def fit_scaler(train_values):
    train_values = np.asarray(train_values, dtype=float)
    if train_values.ndim != 1 or len(train_values) == 0 or not np.isfinite(train_values).all():
        raise ValueError("Scaler needs a nonempty finite one-dimensional training series")
    mean = float(np.mean(train_values))
    std = float(np.std(train_values, ddof=0))
    return {"mean": mean, "std": std if std > 0 else 1.0}


def scale_values(values, scaler):
    return (np.asarray(values, dtype=float) - scaler["mean"]) / scaler["std"]


def inverse_scale(values, scaler):
    return np.asarray(values, dtype=float) * scaler["std"] + scaler["mean"]

In [29]:
def make_windows(scaled_values, lookback, start, end):
    scaled_values = np.asarray(scaled_values, dtype=float)
    if scaled_values.ndim != 1 or not (1 <= lookback <= start < end <= len(scaled_values)):
        raise ValueError("Invalid series shape, lookback, or target range")
    X = np.empty((end - start, lookback, 1), dtype=np.float32)
    y = np.empty((end - start, 1), dtype=np.float32)
    positions = np.arange(start, end, dtype=np.int64)
    for i, t in enumerate(positions):
        X[i, :, 0] = scaled_values[t - lookback:t]
        y[i, 0] = scaled_values[t]
    return X, y, positions

In [30]:
def prepare_fold(values, lookback, train_end, val_end):
    values = np.asarray(values, dtype=float)
    if not (lookback < train_end < val_end <= len(values)):
        raise ValueError("Invalid fold boundaries")
    scaler = fit_scaler(values[:train_end])
    scaled = scale_values(values[:val_end], scaler)
    X_train, y_train, train_positions = make_windows(scaled, lookback, lookback, train_end)
    X_val, y_val, val_positions = make_windows(scaled, lookback, train_end, val_end)
    return {
        "scaler": scaler,
        "X_train": X_train, "y_train": y_train, "train_positions": train_positions,
        "X_val": X_val, "y_val": y_val, "val_positions": val_positions,
    }

In [31]:
def make_loader(X, y, batch_size, shuffle, seed):
    data = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                         torch.as_tensor(y, dtype=torch.float32))
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle,
                      generator=generator, num_workers=0, drop_last=False)

In [32]:
class ElmanRNN(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x_to_h = nn.Linear(1, hidden_size)
        self.h_to_h = nn.Linear(hidden_size, hidden_size, bias=False)
        self.h_to_y = nn.Linear(hidden_size, 1)

    def forward(self, X):
        if X.ndim != 3 or X.shape[1] < 1 or X.shape[2] != 1:
            raise ValueError("X must have shape (batch, sequence_length, 1)")

        hidden = X.new_zeros((X.shape[0], self.hidden_size))
        for step in range(X.shape[1]):
            x_step = X[:, step, :]
            hidden = torch.tanh(self.x_to_h(x_step) + self.h_to_h(hidden))

        return self.h_to_y(hidden)

In [33]:
class JordanRNN(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x_to_h = nn.Linear(1, hidden_size)
        self.y_to_h = nn.Linear(1, hidden_size, bias=False)
        self.h_to_y = nn.Linear(hidden_size, 1)

    def forward(self, X):
        if X.ndim != 3 or X.shape[1] < 1 or X.shape[2] != 1:
            raise ValueError("X must have shape (batch, sequence_length, 1)")

        previous_output = X.new_zeros((X.shape[0], 1))
        for step in range(X.shape[1]):
            x_step = X[:, step, :]
            hidden = torch.tanh(self.x_to_h(x_step) + self.y_to_h(previous_output))
            previous_output = self.h_to_y(hidden)

        return previous_output

In [34]:
class MultiRecurrentRNN(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x_to_h = nn.Linear(1, hidden_size)
        self.h_to_h = nn.Linear(hidden_size, hidden_size, bias=False)
        self.y_to_h = nn.Linear(1, hidden_size, bias=False)
        self.h_to_y = nn.Linear(hidden_size, 1)

    def forward(self, X):
        if X.ndim != 3 or X.shape[1] < 1 or X.shape[2] != 1:
            raise ValueError("X must have shape (batch, sequence_length, 1)")
        hidden = X.new_zeros((X.shape[0], self.hidden_size))
        previous_output = X.new_zeros((X.shape[0], 1))
        for step in range(X.shape[1]):
            x_step = X[:, step, :]
            hidden = torch.tanh(self.x_to_h(x_step) + self.h_to_h(hidden)
                                + self.y_to_h(previous_output))
            previous_output = self.h_to_y(hidden)
        return previous_output


def build_model(kind, hidden_size):
    models = {"elman": ElmanRNN, "jordan": JordanRNN,
              "multi": MultiRecurrentRNN}
    if kind not in models:
        raise ValueError(f"Unknown model kind: {kind}")
    return models[kind](hidden_size)

In [35]:
def regression_metrics(y_true, y_pred):
    actual = np.asarray(y_true, dtype=float).reshape(-1)
    predicted = np.asarray(y_pred, dtype=float).reshape(-1)
    if len(actual) == 0 or len(actual) != len(predicted):
        raise ValueError("Metrics need nonempty matching arrays")
    if not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError("Metrics need finite arrays")
    errors = predicted - actual
    return {"rmse": float(np.sqrt(np.mean(errors ** 2))),
            "mae": float(np.mean(np.abs(errors)))}


def baseline_predictions(values, positions, lag=1):
    values = np.asarray(values, dtype=float)
    positions = np.asarray(positions, dtype=np.int64)
    if values.ndim != 1 or positions.ndim != 1 or len(positions) == 0:
        raise ValueError("Values and positions must be one-dimensional and nonempty")
    if lag < 1 or (positions - lag < 0).any() or (positions >= len(values)).any():
        raise ValueError("Baseline references an unavailable observation")
    return values[positions - lag]


SEASONAL_LAGS = {"Bikeshare": 7, "US electricity": 12,
                 "US gasoline": 52, "ETTh1": 24}

In [36]:
def train_epoch(model, loader, optimizer, loss_fn, clip_norm):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        predicted = model(X_batch)
        if predicted.shape != y_batch.shape:
            raise ValueError("Prediction and target shapes differ")
        loss = loss_fn(predicted, y_batch)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training loss")
        loss.backward()
        gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        if not torch.isfinite(gradient_norm):
            raise FloatingPointError("Non-finite gradient norm")
        optimizer.step()
        count = len(X_batch)
        total_loss += loss.item() * count
        total_samples += count
    if total_samples == 0:
        raise ValueError("Empty training loader")
    return total_loss / total_samples


def predict_scaled(model, loader):
    model.eval()
    pieces = []
    with torch.no_grad():
        for X_batch, _ in loader:
            pieces.append(model(X_batch.to(DEVICE)).cpu().numpy())
    if not pieces:
        raise ValueError("Empty prediction loader")
    return np.concatenate(pieces, axis=0)

In [37]:
def fit_with_early_stopping(model, train_loader, train_eval_loader, val_loader,
                            y_train, y_val, config):
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                 weight_decay=config["weight_decay"])
    loss_fn = nn.MSELoss()
    best_mse = float("inf")
    best_state = None
    best_epoch = 0
    bad_epochs = 0
    history_rows = []
    for epoch in range(1, config["max_epochs"] + 1):
        started = time.perf_counter()
        train_epoch(model, train_loader, optimizer, loss_fn, config["clip_norm"])
        train_pred = predict_scaled(model, train_eval_loader)
        val_pred = predict_scaled(model, val_loader)
        train_mse = float(np.mean((train_pred - y_train) ** 2))
        val_mse = float(np.mean((val_pred - y_val) ** 2))
        if not np.isfinite([train_mse, val_mse]).all():
            raise FloatingPointError("Non-finite evaluation loss")
        history_rows.append({"epoch": epoch, "train_mse": train_mse,
                             "val_mse": val_mse,
                             "seconds": time.perf_counter() - started})
        if val_mse < best_mse:
            best_mse = val_mse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
        if bad_epochs >= config["patience"]:
            break
    model.load_state_dict(best_state)
    model.eval()
    history = pd.DataFrame(history_rows)
    assert best_epoch == int(history.loc[history.val_mse.idxmin(), "epoch"])
    restored_mse = float(np.mean((predict_scaled(model, val_loader) - y_val) ** 2))
    assert np.isclose(restored_mse, best_mse, rtol=1e-5)
    return {"model": model, "history": history, "best_epoch": best_epoch,
            "epochs_ran": len(history)}

In [38]:
def run_fold(name, kind, hidden_size, fold, seed, config):
    values = series_values[name]
    lookback = LOOKBACKS[name]
    data = prepare_fold(values, lookback, fold["train_end"], fold["val_end"])
    set_seed(seed)
    model = build_model(kind, hidden_size).to(DEVICE)
    train_loader = make_loader(data["X_train"], data["y_train"],
                               config["batch_size"], True, seed)
    train_eval = make_loader(data["X_train"], data["y_train"],
                             config["batch_size"], False, seed)
    val_loader = make_loader(data["X_val"], data["y_val"],
                             config["batch_size"], False, seed)
    started = time.perf_counter()
    fit = fit_with_early_stopping(model, train_loader, train_eval, val_loader,
                                  data["y_train"], data["y_val"], config)
    seconds = time.perf_counter() - started
    predictions = inverse_scale(predict_scaled(model, val_loader)[:, 0], data["scaler"])
    positions = data["val_positions"]
    actual = values[positions]
    np.testing.assert_allclose(inverse_scale(data["y_val"][:, 0], data["scaler"]),
                               actual, rtol=1e-5, atol=1e-5)
    scores = regression_metrics(actual, predictions)
    persistence = regression_metrics(actual, baseline_predictions(values, positions, 1))
    seasonal = (regression_metrics(actual, baseline_predictions(values, positions,
                SEASONAL_LAGS[name])) if name in SEASONAL_LAGS else
                {"rmse": np.nan, "mae": np.nan})
    row = {"dataset": name, "model": kind, "hidden_size": hidden_size,
           "fold": fold["fold"], "seed": seed, "train_end": fold["train_end"],
           "val_end": fold["val_end"], "n_train": len(data["y_train"]),
           "n_val": len(data["y_val"]),
           "parameters": sum(p.numel() for p in model.parameters()),
           "best_epoch": fit["best_epoch"], "epochs_ran": fit["epochs_ran"],
           "seconds": seconds, **scores,
           "persistence_rmse": persistence["rmse"],
           "persistence_mae": persistence["mae"],
           "seasonal_rmse": seasonal["rmse"], "seasonal_mae": seasonal["mae"]}
    return {"row": row, "history": fit["history"], "model": model,
            "scaler": data["scaler"], "positions": positions,
            "predictions": predictions}

In [39]:
HISTORY_DIR = OUTPUT_DIR / "histories"
FIGURE_DIR = OUTPUT_DIR / "figures"
for directory in (HISTORY_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def file_slug(name):
    return name.lower().replace(" ", "_")


cv_rows = []
cv_started = time.perf_counter()
cv_expected = len(series_values) * len(MODEL_NAMES) * len(CONFIG["hidden_sizes"]) * 3
for name in series_values:
    for kind in MODEL_NAMES:
        for hidden_size in CONFIG["hidden_sizes"]:
            for fold in make_folds(dev_ends[name]):
                outcome = run_fold(name, kind, hidden_size, fold,
                                   CONFIG["tuning_seed"], CONFIG)
                cv_rows.append(outcome["row"])
                pd.DataFrame(cv_rows).to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
                filename = (f"cv_{file_slug(name)}_{kind}_h{hidden_size}"
                            f"_f{fold['fold']}_s{CONFIG['tuning_seed']}.csv")
                outcome["history"].to_csv(HISTORY_DIR / filename, index=False)
                print(f"CV {len(cv_rows)}/{cv_expected}: {name}, {kind}, "
                      f"H={hidden_size}, fold={fold['fold']}, "
                      f"RMSE={outcome['row']['rmse']:.4f}, "
                      f"elapsed={time.perf_counter()-cv_started:.1f}s")
                del outcome

cv_results = pd.DataFrame(cv_rows)
cv_elapsed = time.perf_counter() - cv_started
assert len(cv_results) == cv_expected
assert not cv_results.duplicated(["dataset", "model", "hidden_size", "fold", "seed"]).any()
assert all(row.val_end <= dev_ends[row.dataset] for row in cv_results.itertuples())

cv_summary = (cv_results.groupby(["dataset", "model", "hidden_size"], as_index=False)
              .agg(mean_rmse=("rmse", "mean"), sd_rmse=("rmse", "std"),
                   mean_mae=("mae", "mean"), sd_mae=("mae", "std"),
                   mean_best_epoch=("best_epoch", "mean"),
                   mean_seconds=("seconds", "mean")))
cv_summary.to_csv(OUTPUT_DIR / "cv_summary.csv", index=False)

selected_configs = {}
for name in series_values:
    for kind in MODEL_NAMES:
        candidates = cv_summary.query("dataset == @name and model == @kind")
        chosen = candidates.sort_values(["mean_rmse", "hidden_size"]).iloc[0]
        width = int(chosen.hidden_size)
        matching = cv_results.query(
            "dataset == @name and model == @kind and hidden_size == @width"
        ).sort_values("fold")
        best_epochs = matching.best_epoch.astype(int).tolist()
        selected_configs[(name, kind)] = {
            "dataset": name, "model": kind, "hidden_size": width,
            "cv_rmse": float(chosen.mean_rmse),
            "cv_mae": float(chosen.mean_mae),
            "best_epochs": best_epochs,
            "final_epochs": max(1, int(np.median(best_epochs))),
        }
assert len(selected_configs) == len(series_values) * len(MODEL_NAMES)
with open(OUTPUT_DIR / "selected_configs.json", "w", encoding="utf-8") as handle:
    json.dump(list(selected_configs.values()), handle, indent=2)

development_winners = (pd.DataFrame(selected_configs.values())
                       .sort_values(["dataset", "cv_rmse", "model"])
                       .groupby("dataset", as_index=False).first())
display(cv_summary, development_winners)

CV 1/90: Bikeshare, elman, H=8, fold=1, RMSE=1046.4463, elapsed=2.3s
CV 2/90: Bikeshare, elman, H=8, fold=2, RMSE=1539.4342, elapsed=3.6s
CV 3/90: Bikeshare, elman, H=8, fold=3, RMSE=1467.8525, elapsed=5.1s
CV 4/90: Bikeshare, elman, H=16, fold=1, RMSE=1026.8218, elapsed=5.4s
CV 5/90: Bikeshare, elman, H=16, fold=2, RMSE=1390.1229, elapsed=6.5s
CV 6/90: Bikeshare, elman, H=16, fold=3, RMSE=1136.5726, elapsed=7.8s
CV 7/90: Bikeshare, jordan, H=8, fold=1, RMSE=812.9938, elapsed=8.8s
CV 8/90: Bikeshare, jordan, H=8, fold=2, RMSE=1309.8508, elapsed=10.4s
CV 9/90: Bikeshare, jordan, H=8, fold=3, RMSE=1279.1904, elapsed=12.1s
CV 10/90: Bikeshare, jordan, H=16, fold=1, RMSE=817.1224, elapsed=12.6s
CV 11/90: Bikeshare, jordan, H=16, fold=2, RMSE=1372.9689, elapsed=13.1s
CV 12/90: Bikeshare, jordan, H=16, fold=3, RMSE=1333.2703, elapsed=14.9s
CV 13/90: Bikeshare, multi, H=8, fold=1, RMSE=828.9995, elapsed=15.8s
CV 14/90: Bikeshare, multi, H=8, fold=2, RMSE=1322.4444, elapsed=16.9s
CV 15/90: Bik

,dataset,model,hidden_size,mean_rmse,sd_rmse,mean_mae,sd_mae,mean_best_epoch,mean_seconds
0,Bikeshare,elman,8,1351.244329,266.378250,1134.117883,307.802641,53.000000,1.663842
1,Bikeshare,elman,16,1184.505798,186.333343,977.087433,188.228620,37.333333,0.915035
2,Bikeshare,jordan,8,1134.011653,278.431988,950.608252,264.154792,59.333333,1.409857
3,Bikeshare,jordan,16,1174.453864,310.094047,996.006911,287.942915,32.000000,0.925514
4,Bikeshare,multi,8,1136.416487,268.185470,942.138911,256.913150,42.000000,1.306983
5,Bikeshare,multi,16,1131.242030,295.125982,941.734961,276.928809,33.000000,1.216087
6,ETTh1,elman,8,0.812063,0.216964,0.546935,0.120986,32.666667,21.476540
7,ETTh1,elman,16,0.807895,0.213425,0.545810,0.117222,14.000000,12.051745
8,ETTh1,jordan,8,0.810245,0.205943,0.552061,0.114344,41.000000,32.827914
9,ETTh1,jordan,16,0.815016,0.206530,0.552690,0.111420,17.333333,17.585101


,dataset,model,hidden_size,cv_rmse,cv_mae,best_epochs,final_epochs
0,Bikeshare,multi,16,1131.242030,941.734961,"[1, 49, 49]",49
1,ETTh1,elman,16,0.807895,0.545810,"[20, 9, 13]",13
2,Sunspots,elman,16,13.682326,9.940660,"[60, 27, 27]",27
3,US electricity,multi,16,30.512491,23.835539,"[60, 60, 60]",60
4,US gasoline,multi,16,0.249089,0.204756,"[55, 49, 7]",49


In [40]:
diagnostic_rows = []
for (name, kind), choice in selected_configs.items():
    width = choice["hidden_size"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
    for fold_number, ax in enumerate(axes, start=1):
        filename = (f"cv_{file_slug(name)}_{kind}_h{width}_f{fold_number}"
                    f"_s{CONFIG['tuning_seed']}.csv")
        history = pd.read_csv(HISTORY_DIR / filename)
        ax.plot(history.epoch, history.train_mse, label="train MSE")
        ax.plot(history.epoch, history.val_mse, label="validation MSE")
        best = int(history.loc[history.val_mse.idxmin(), "epoch"])
        ax.axvline(best, color="black", linestyle=":", linewidth=1)
        ax.set(title=f"Fold {fold_number}", xlabel="epoch", ylabel="scaled MSE")
        ax.grid(alpha=0.2)
        if fold_number == 1:
            ax.legend()
        last = history.iloc[-1]
        first = history.iloc[0]
        if last.train_mse < first.train_mse and last.val_mse > history.val_mse.min() * 1.1:
            observation = "validation worsened after its best epoch"
            response = "best validation checkpoint restored; compare smaller width"
        elif best == CONFIG["max_epochs"]:
            observation = "validation was best at the epoch cap"
            response = "consider a shared larger cap on development data"
        elif last.train_mse >= first.train_mse * 0.95:
            observation = "training loss improved little"
            response = "check gradients, width and baseline on development data"
        else:
            observation = "learning curve improved before stopping"
            response = "retain the selected configuration"
        diagnostic_rows.append({
            "dataset": name, "model": kind, "hidden_size": width,
            "fold": fold_number, "best_epoch": best,
            "first_train_mse": float(first.train_mse),
            "last_train_mse": float(last.train_mse),
            "best_val_mse": float(history.val_mse.min()),
            "last_val_mse": float(last.val_mse),
            "observation": observation, "response": response,
            "review_needed": observation != "learning curve improved before stopping",
        })
    fig.suptitle(f"{name}: {kind}, H={width}")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"cv_curves_{file_slug(name)}_{kind}.png", dpi=140)
    plt.close(fig)

diagnostics = pd.DataFrame(diagnostic_rows)
diagnostics.to_csv(OUTPUT_DIR / "diagnostics.csv", index=False)
display(diagnostics)

,dataset,model,hidden_size,fold,best_epoch,first_train_mse,last_train_mse,best_val_mse,last_val_mse,observation,response,review_needed
0,Bikeshare,elman,16,1,14,0.964058,0.285936,0.526676,0.552924,learning curve improved before stopping,retain the selected configuration,False
1,Bikeshare,elman,16,2,49,0.967171,0.302549,1.085137,1.086119,learning curve improved before stopping,retain the selected configuration,False
2,Bikeshare,elman,16,3,49,0.971536,0.280724,0.556566,0.557147,learning curve improved before stopping,retain the selected configuration,False
3,Bikeshare,jordan,8,1,60,1.888044,0.403322,0.330162,0.330162,validation was best at the epoch cap,consider a shared larger cap on development data,True
4,Bikeshare,jordan,8,2,58,1.889095,0.363405,0.963434,0.966676,learning curve improved before stopping,retain the selected configuration,False
5,Bikeshare,jordan,8,3,60,1.883921,0.345300,0.705006,0.705006,validation was best at the epoch cap,consider a shared larger cap on development data,True
6,Bikeshare,multi,16,1,1,0.655090,0.296989,0.337656,0.504582,validation worsened after its best epoch,best validation checkpoint restored; compare s...,True
7,Bikeshare,multi,16,2,49,0.652825,0.293115,1.116541,1.173930,learning curve improved before stopping,retain the selected configuration,False
8,Bikeshare,multi,16,3,49,0.663958,0.282869,0.581210,0.585947,learning curve improved before stopping,retain the selected configuration,False
9,Sunspots,elman,16,1,60,0.855946,0.152088,0.109531,0.109531,validation was best at the epoch cap,consider a shared larger cap on development data,True


In [41]:
def fit_fixed_epochs(model, train_loader, config, epochs):
    if epochs < 1:
        raise ValueError("At least one epoch is required")
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                 weight_decay=config["weight_decay"])
    loss_fn = nn.MSELoss()
    rows = []
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        mse = train_epoch(model, train_loader, optimizer, loss_fn, config["clip_norm"])
        rows.append({"epoch": epoch, "train_mse": mse,
                     "seconds": time.perf_counter() - started})
    model.eval()
    return model, pd.DataFrame(rows)


def run_final(name, kind, hidden_size, epochs, seed, config):
    values = series_values[name]
    D = dev_ends[name]
    lookback = LOOKBACKS[name]
    scaler = fit_scaler(values[:D])
    scaled = scale_values(values, scaler)
    X_train, y_train, train_positions = make_windows(scaled, lookback, lookback, D)
    X_test, y_test, test_positions = make_windows(scaled, lookback, D, len(values))
    assert train_positions.max() < D and test_positions.min() == D
    set_seed(seed)
    model = build_model(kind, hidden_size).to(DEVICE)
    train_loader = make_loader(X_train, y_train, config["batch_size"], True, seed)
    test_loader = make_loader(X_test, y_test, config["batch_size"], False, seed)
    started = time.perf_counter()
    model, history = fit_fixed_epochs(model, train_loader, config, epochs)
    seconds = time.perf_counter() - started
    predictions = inverse_scale(predict_scaled(model, test_loader)[:, 0], scaler)
    actual = values[test_positions]
    np.testing.assert_allclose(inverse_scale(y_test[:, 0], scaler), actual,
                               rtol=1e-5, atol=1e-5)
    scores = regression_metrics(actual, predictions)
    persistence = regression_metrics(actual, baseline_predictions(values, test_positions, 1))
    seasonal = (regression_metrics(actual, baseline_predictions(values, test_positions,
                SEASONAL_LAGS[name])) if name in SEASONAL_LAGS else
                {"rmse": np.nan, "mae": np.nan})
    row = {"dataset": name, "model": kind, "hidden_size": hidden_size,
           "lookback": lookback, "seed": seed, "final_epochs": epochs,
           "development_end": D, "n_train": len(y_train), "n_test": len(y_test),
           "parameters": sum(p.numel() for p in model.parameters()),
           "seconds": seconds, **scores,
           "persistence_rmse": persistence["rmse"],
           "persistence_mae": persistence["mae"],
           "seasonal_rmse": seasonal["rmse"], "seasonal_mae": seasonal["mae"]}
    predictions_frame = pd.DataFrame({
        "dataset": name, "model": kind, "seed": seed,
        "target_position": test_positions,
        "actual_index": [str(item) for item in series_index[name][test_positions]],
        "actual": actual, "predicted": predictions,
        "residual": actual - predictions,
    })
    return {"model": model, "scaler": scaler, "history": history,
            "row": row, "predictions": predictions_frame}


test_rows = []
prediction_frames = []
final_started = time.perf_counter()
final_expected = len(selected_configs) * len(CONFIG["final_seeds"])
for (name, kind), choice in selected_configs.items():
    for seed in CONFIG["final_seeds"]:
        outcome = run_final(name, kind, choice["hidden_size"],
                            choice["final_epochs"], seed, CONFIG)
        test_rows.append(outcome["row"])
        prediction_frames.append(outcome["predictions"])
        pd.DataFrame(test_rows).to_csv(OUTPUT_DIR / "test_results.csv", index=False)
        pd.concat(prediction_frames, ignore_index=True).to_csv(
            OUTPUT_DIR / "test_predictions.csv", index=False)
        stem = f"{file_slug(name)}_{kind}_h{choice['hidden_size']}_s{seed}"
        outcome["history"].to_csv(HISTORY_DIR / f"final_{stem}.csv", index=False)
        print(f"Final {len(test_rows)}/{final_expected}: {name}, {kind}, "
              f"seed={seed}, RMSE={outcome['row']['rmse']:.4f}, "
              f"elapsed={time.perf_counter()-final_started:.1f}s")
        del outcome

test_results = pd.DataFrame(test_rows)
test_predictions = pd.concat(prediction_frames, ignore_index=True)
final_elapsed = time.perf_counter() - final_started
assert len(test_results) == final_expected
assert not test_results.duplicated(["dataset", "model", "seed"]).any()
assert all(row.n_test == len(series_values[row.dataset]) - dev_ends[row.dataset]
           for row in test_results.itertuples())

Final 1/45: Bikeshare, elman, seed=11, RMSE=1269.9156, elapsed=0.9s
Final 2/45: Bikeshare, elman, seed=22, RMSE=1255.9144, elapsed=1.9s
Final 3/45: Bikeshare, elman, seed=33, RMSE=1244.5978, elapsed=2.7s
Final 4/45: Bikeshare, jordan, seed=11, RMSE=1274.4329, elapsed=4.1s
Final 5/45: Bikeshare, jordan, seed=22, RMSE=1268.0616, elapsed=5.5s
Final 6/45: Bikeshare, jordan, seed=33, RMSE=1252.6456, elapsed=6.7s
Final 7/45: Bikeshare, multi, seed=11, RMSE=1255.2799, elapsed=8.0s
Final 8/45: Bikeshare, multi, seed=22, RMSE=1282.1403, elapsed=9.4s
Final 9/45: Bikeshare, multi, seed=33, RMSE=1237.7994, elapsed=10.6s
Final 10/45: Sunspots, elman, seed=11, RMSE=21.3047, elapsed=13.1s
Final 11/45: Sunspots, elman, seed=22, RMSE=21.4948, elapsed=23.3s
Final 12/45: Sunspots, elman, seed=33, RMSE=21.8514, elapsed=33.9s
Final 13/45: Sunspots, jordan, seed=11, RMSE=20.7911, elapsed=53.3s
Final 14/45: Sunspots, jordan, seed=22, RMSE=21.1393, elapsed=75.7s
Final 15/45: Sunspots, jordan, seed=33, RMSE=21

In [42]:
units = {"Bikeshare": "rented bikes/day", "Sunspots": "sunspots/month",
         "US electricity": "billion kWh/month", "US gasoline": "million barrels/day",
         "ETTh1": "oil temperature (degrees C)"}
final_summary = (test_results.groupby(["dataset", "model"], as_index=False)
                 .agg(hidden_size=("hidden_size", "first"),
                      lookback=("lookback", "first"),
                      parameters=("parameters", "first"),
                      final_epochs=("final_epochs", "first"),
                      mean_rmse=("rmse", "mean"), sd_rmse=("rmse", "std"),
                      mean_mae=("mae", "mean"), sd_mae=("mae", "std"),
                      mean_seconds=("seconds", "mean"),
                      persistence_rmse=("persistence_rmse", "first"),
                      persistence_mae=("persistence_mae", "first"),
                      seasonal_rmse=("seasonal_rmse", "first"),
                      seasonal_mae=("seasonal_mae", "first")))
final_summary = final_summary.merge(
    pd.DataFrame(selected_configs.values())[["dataset", "model", "cv_rmse"]],
    on=["dataset", "model"], validate="one_to_one")
final_summary["test_rank"] = (final_summary.groupby("dataset").mean_rmse
                              .rank(method="min").astype(int))
final_summary["development_rank"] = (final_summary.groupby("dataset").cv_rmse
                                     .rank(method="min").astype(int))
final_summary.to_csv(OUTPUT_DIR / "final_summary.csv", index=False)
assert len(final_summary) == len(series_values) * len(MODEL_NAMES)
display(final_summary.sort_values(["dataset", "test_rank"]))

for name in series_values:
    subset = final_summary[final_summary.dataset == name].set_index("model").loc[list(MODEL_NAMES)]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(np.arange(3), subset.mean_rmse, yerr=subset.sd_rmse, capsize=4)
    ax.axhline(float(subset.persistence_rmse.iloc[0]), color="black",
               linestyle="--", label="persistence")
    if name in SEASONAL_LAGS:
        ax.axhline(float(subset.seasonal_rmse.iloc[0]), color="tab:red",
                   linestyle=":", label="seasonal naive")
    ax.set(xticks=np.arange(3), xticklabels=MODEL_NAMES,
           ylabel=f"Test RMSE ({units[name]})", title=f"{name}: test RMSE")
    ax.grid(axis="y", alpha=0.2)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"rmse_{file_slug(name)}.png", dpi=140)
    plt.close(fig)

    chosen_seed = CONFIG["final_seeds"][0]
    plotted = test_predictions.query("dataset == @name and seed == @chosen_seed")
    actual_frame = plotted[plotted.model == MODEL_NAMES[0]].sort_values("target_position")
    positions = actual_frame.target_position.to_numpy(dtype=int)
    x_axis = series_index[name][positions]
    actual = actual_frame.actual.to_numpy()
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(x_axis, actual, color="black", linewidth=1.2, label="actual")
    ax.plot(x_axis, baseline_predictions(series_values[name], positions),
            color="gray", linewidth=0.8, label="persistence")
    for kind in MODEL_NAMES:
        frame = plotted[plotted.model == kind].sort_values("target_position")
        np.testing.assert_array_equal(frame.target_position.to_numpy(), positions)
        ax.plot(x_axis, frame.predicted, linewidth=0.8, label=kind)
    ax.set(title=f"{name}: held-out one-step forecasts (seed {chosen_seed})",
           ylabel=units[name], xlabel="time")
    ax.legend(ncol=5, fontsize=8)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"forecast_{file_slug(name)}.png", dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 3.4))
    for kind in MODEL_NAMES:
        frame = plotted[plotted.model == kind].sort_values("target_position")
        ax.plot(x_axis, frame.residual, linewidth=0.8, label=kind)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=f"{name}: test residuals (seed {chosen_seed})",
           ylabel=f"actual - prediction ({units[name]})", xlabel="time")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"residual_{file_slug(name)}.png", dpi=140)
    plt.close(fig)

test_winners = (final_summary.sort_values(["dataset", "mean_rmse", "model"])
                .groupby("dataset", as_index=False).first())
display(test_winners[["dataset", "model", "mean_rmse", "mean_mae",
                      "development_rank", "persistence_rmse", "seasonal_rmse"]])

,dataset,model,hidden_size,lookback,parameters,final_epochs,mean_rmse,sd_rmse,mean_mae,sd_mae,mean_seconds,persistence_rmse,persistence_mae,seasonal_rmse,seasonal_mae,cv_rmse,test_rank,development_rank
0,Bikeshare,elman,16,14,305,49,1256.809277,12.682566,960.428942,10.421533,0.904358,1282.315269,878.394558,1759.655544,1194.734694,1184.505798,1,3
2,Bikeshare,multi,16,14,321,49,1258.406530,22.335223,958.792696,31.397276,1.281149,1282.315269,878.394558,1759.655544,1194.734694,1131.242030,2,1
1,Bikeshare,jordan,8,14,33,60,1265.046705,11.202153,996.189232,25.794807,1.315765,1282.315269,878.394558,1759.655544,1194.734694,1134.011653,3,2
4,ETTh1,jordan,8,24,33,34,0.658052,0.004173,0.457021,0.006203,55.438383,0.654184,0.448086,2.236520,1.713828,0.810245,1,3
3,ETTh1,elman,16,24,305,13,0.680360,0.045178,0.481880,0.052232,17.976023,0.654184,0.448086,2.236520,1.713828,0.807895,2,1
5,ETTh1,multi,16,24,321,13,0.684051,0.050114,0.485660,0.056834,26.138894,0.654184,0.448086,2.236520,1.713828,0.808883,3,2
7,Sunspots,jordan,16,24,65,59,21.043978,0.221204,15.036871,0.109261,20.628158,20.090704,14.836879,NaN,NaN,13.831870,1,3
8,Sunspots,multi,16,24,321,34,21.263004,0.120189,15.174303,0.032430,14.877718,20.090704,14.836879,NaN,NaN,13.751169,2,2
6,Sunspots,elman,16,24,305,27,21.550265,0.277542,15.215583,0.061819,7.672601,20.090704,14.836879,NaN,NaN,13.682326,3,1
11,US electricity,multi,16,24,321,60,30.652773,0.722471,24.207528,0.528474,3.810340,32.549677,27.401102,13.729415,10.668714,30.512491,1,1


,dataset,model,mean_rmse,mean_mae,development_rank,persistence_rmse,seasonal_rmse
0,Bikeshare,elman,1256.809277,960.428942,3,1282.315269,1759.655544
1,ETTh1,jordan,0.658052,0.457021,3,0.654184,2.236520
2,Sunspots,jordan,21.043978,15.036871,3,20.090704,NaN
3,US electricity,multi,30.652773,24.207528,1,32.549677,13.729415
4,US gasoline,multi,0.283446,0.225506,1,0.321766,0.404405


In [43]:
assert len(data_audit) == 5
assert len(stationarity_results) == 5
assert len(cv_results) == len(series_values) * len(MODEL_NAMES) * len(CONFIG["hidden_sizes"]) * 3
assert len(selected_configs) == len(series_values) * len(MODEL_NAMES)
assert len(test_results) == len(selected_configs) * len(CONFIG["final_seeds"])
assert not cv_results.duplicated(["dataset", "model", "hidden_size", "fold", "seed"]).any()
assert not test_results.duplicated(["dataset", "model", "seed"]).any()
for name, values in series_values.items():
    D = dev_ends[name]
    assert cv_results.loc[cv_results.dataset == name, "val_end"].max() == D
    for kind in MODEL_NAMES:
        for seed in CONFIG["final_seeds"]:
            frame = test_predictions.query(
                "dataset == @name and model == @kind and seed == @seed"
            ).sort_values("target_position")
            np.testing.assert_array_equal(frame.target_position, np.arange(D, len(values)))
            np.testing.assert_allclose(frame.actual, values[D:])
            np.testing.assert_allclose(frame.residual, frame.actual - frame.predicted)